# Trial-wise responses analysis - CLMM for acceptance

- [ ] Acceptance: cumulative link mixed model

In [42]:
library(utils)
library(ordinal)
library(emmeans)
library(dplyr)
library(tidyverse)   # read_csv, dplyr, ggplot2, etc.
library(DHARMa)

This is DHARMa 0.5.0. For overview type '?DHARMa'. For recent changes, type news(package = 'DHARMa') 

Note that the default setting in simulateResiduals was changed to conditional simulations since version 0.5.0. This is likely to change residual calculations for all hierarchical (in particular random effect) models. If you want to switch back to the old package version defaults, please use the argument simulateREs = "user-specified" in simulateResiduals(). For more details, see ?simulateREsiduals.



In [43]:
# Single ordinal response analysed using CLMM
li_resp_CLMM <- c("Accept", "Authorship")

wide <- read_csv(
  "data_analysis/trial_wise_SoA_with_pp_traits.csv",
  show_col_types = FALSE
)
head(wide, 3)

pp,trial,voice_id,condition,scene_id,1,2,3,4,5,⋯,8,9,ans_sd,SoPA,SoNA,Accept,SoA,Authorship,AI_literacy_z,DoC_z
<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
P01,1,clone,repeat,1,4,6,1,4,2,⋯,2,3,1.752549,2.8,4.5,3,3.000000,4,1.370731,-0.06454751
P01,2,clone,enhance,3,2,2,3,2,1,⋯,3,7,1.832251,2.0,5.5,7,2.142857,2,1.370731,-0.06454751
P01,3,clone,counter,5,1,2,2,1,2,⋯,3,4,2.375470,1.6,6.0,4,1.714286,1,1.370731,-0.06454751


In [44]:
wide$Accept <- ordered(wide$Accept)  # ordered
wide$Authorship <- ordered(wide$Authorship)  # ordered

wide$condition <- factor(
    wide$condition,
    levels=c("repeat","enhance","counter")
)
head(wide, 3)

pp,trial,voice_id,condition,scene_id,1,2,3,4,5,⋯,8,9,ans_sd,SoPA,SoNA,Accept,SoA,Authorship,AI_literacy_z,DoC_z
<chr>,<dbl>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<ord>,<dbl>,<ord>,<dbl>,<dbl>
P01,1,clone,repeat,1,4,6,1,4,2,⋯,2,3,1.752549,2.8,4.5,3,3.000000,4,1.370731,-0.06454751
P01,2,clone,enhance,3,2,2,3,2,1,⋯,3,7,1.832251,2.0,5.5,7,2.142857,2,1.370731,-0.06454751
P01,3,clone,counter,5,1,2,2,1,2,⋯,3,4,2.375470,1.6,6.0,4,1.714286,1,1.370731,-0.06454751


# CLMM - acceptance
SoA ~ condition * voice_id + AI_literacy_z + DoC_z + (1 | pp)

In [49]:
# 2
model_accept <- clmm(
    Accept ~
        voice_id * condition +
        AI_literacy_z +
        DoC_z +
        (1|pp),
    data=wide,
)

# 3
summary(model_accept)

# 4
joint_tests(model_accept)

# 5
emm <- emmeans(model_accept,
               ~ voice_id * condition,
            mode="latent")

pairs(emm, adjust="holm")

Cumulative Link Mixed Model fitted with the Laplace approximation

formula: Accept ~ voice_id * condition + AI_literacy_z + DoC_z + (1 |      pp)
data:    wide

 link  threshold nobs logLik  AIC     niter      max.grad cond.H 
 logit flexible  326  -662.74 1359.48 2012(6039) 1.25e-03 4.2e+02

Random effects:
 Groups Name        Variance Std.Dev.
 pp     (Intercept) 0.6625   0.8139  
Number of groups:  pp 28 

Coefficients:
                                 Estimate Std. Error z value Pr(>|z|)    
voice_idrobotic                   0.54332    0.35285   1.540   0.1236    
conditionenhance                 -0.67905    0.33060  -2.054   0.0400 *  
conditioncounter                 -2.56938    0.36243  -7.089 1.35e-12 ***
AI_literacy_z                     0.17605    0.19270   0.914   0.3609    
DoC_z                            -0.08559    0.19421  -0.441   0.6594    
voice_idrobotic:conditionenhance  0.05252    0.48237   0.109   0.9133    
voice_idrobotic:conditioncounter -0.85281    0.49952  -

,model term,df1,df2,F.ratio,Chisq,p.value
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,voice_id,1,Inf,1.914,1.914,1.664837e-01
3,condition,2,Inf,56.240,112.480,3.762025e-25
4,AI_literacy_z,1,Inf,0.835,0.835,3.609367e-01
5,DoC_z,1,Inf,0.194,0.194,6.594463e-01
2,voice_id:condition,2,Inf,2.115,4.230,1.205892e-01


 contrast                          estimate    SE  df z.ratio p.value
 clone repeat - robotic repeat      -0.5433 0.353 Inf  -1.540  0.3708
 clone repeat - clone enhance        0.6790 0.331 Inf   2.054  0.2399
 clone repeat - robotic enhance      0.0832 0.339 Inf   0.245  0.8061
 clone repeat - clone counter        2.5694 0.362 Inf   7.089 <0.0001
 clone repeat - robotic counter      2.8789 0.385 Inf   7.480 <0.0001
 robotic repeat - clone enhance      1.2224 0.349 Inf   3.500  0.0033
 robotic repeat - robotic enhance    0.6265 0.355 Inf   1.763  0.3588
 robotic repeat - clone counter      3.1127 0.384 Inf   8.111 <0.0001
 robotic repeat - robotic counter    3.4222 0.405 Inf   8.446 <0.0001
 clone enhance - robotic enhance    -0.5958 0.331 Inf  -1.801  0.3588
 clone enhance - clone counter       1.8903 0.346 Inf   5.459 <0.0001
 clone enhance - robotic counter     2.1998 0.369 Inf   5.964 <0.0001
 robotic enhance - clone counter     2.4862 0.361 Inf   6.879 <0.0001
 robotic enhance - r

# CLMM - authorship

In [50]:
# 2
model_author <- clmm(
    Accept ~
        voice_id * condition +
        AI_literacy_z +
        DoC_z +
        (1|pp),
    data=wide,
)

# 3
summary(model_author)

# 4
joint_tests(model_author)

# 5
emm <- emmeans(model_author,
               ~ voice_id * condition,
            mode="latent")

pairs(emm, adjust="holm")

Cumulative Link Mixed Model fitted with the Laplace approximation

formula: Accept ~ voice_id * condition + AI_literacy_z + DoC_z + (1 |      pp)
data:    wide

 link  threshold nobs logLik  AIC     niter      max.grad cond.H 
 logit flexible  326  -662.74 1359.48 2012(6039) 1.25e-03 4.2e+02

Random effects:
 Groups Name        Variance Std.Dev.
 pp     (Intercept) 0.6625   0.8139  
Number of groups:  pp 28 

Coefficients:
                                 Estimate Std. Error z value Pr(>|z|)    
voice_idrobotic                   0.54332    0.35285   1.540   0.1236    
conditionenhance                 -0.67905    0.33060  -2.054   0.0400 *  
conditioncounter                 -2.56938    0.36243  -7.089 1.35e-12 ***
AI_literacy_z                     0.17605    0.19270   0.914   0.3609    
DoC_z                            -0.08559    0.19421  -0.441   0.6594    
voice_idrobotic:conditionenhance  0.05252    0.48237   0.109   0.9133    
voice_idrobotic:conditioncounter -0.85281    0.49952  -

,model term,df1,df2,F.ratio,Chisq,p.value
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,voice_id,1,Inf,1.914,1.914,1.664837e-01
3,condition,2,Inf,56.240,112.480,3.762025e-25
4,AI_literacy_z,1,Inf,0.835,0.835,3.609367e-01
5,DoC_z,1,Inf,0.194,0.194,6.594463e-01
2,voice_id:condition,2,Inf,2.115,4.230,1.205892e-01


 contrast                          estimate    SE  df z.ratio p.value
 clone repeat - robotic repeat      -0.5433 0.353 Inf  -1.540  0.3708
 clone repeat - clone enhance        0.6790 0.331 Inf   2.054  0.2399
 clone repeat - robotic enhance      0.0832 0.339 Inf   0.245  0.8061
 clone repeat - clone counter        2.5694 0.362 Inf   7.089 <0.0001
 clone repeat - robotic counter      2.8789 0.385 Inf   7.480 <0.0001
 robotic repeat - clone enhance      1.2224 0.349 Inf   3.500  0.0033
 robotic repeat - robotic enhance    0.6265 0.355 Inf   1.763  0.3588
 robotic repeat - clone counter      3.1127 0.384 Inf   8.111 <0.0001
 robotic repeat - robotic counter    3.4222 0.405 Inf   8.446 <0.0001
 clone enhance - robotic enhance    -0.5958 0.331 Inf  -1.801  0.3588
 clone enhance - clone counter       1.8903 0.346 Inf   5.459 <0.0001
 clone enhance - robotic counter     2.1998 0.369 Inf   5.964 <0.0001
 robotic enhance - clone counter     2.4862 0.361 Inf   6.879 <0.0001
 robotic enhance - r